# 01 Ingesta

Baja el zip NOAA del día `fecha`, verifica el archivo y publica el CSV en `landing/`.
`fecha` es un widget para que el job pase el día. Si el CSV ya está en `landing/` y pesa
lo que declara el zip, no se vuelve a bajar.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

El zip se escribe en `raw_zips/_staging/` y solo entonces se renombra a `raw_zips/`.
El CSV hace el mismo camino hacia `landing/`, así el stream no ve un archivo a medias.


In [ ]:
import errno
import os
import shutil
import time
import zipfile
from pathlib import Path

import requests


def mover_en_volume(origen: str, destino: str) -> None:
    """Rename dentro del volume. Si el FUSE no renombra, shutil.move copia y borra."""
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    try:
        os.replace(origen, destino)
    except OSError as exc:
        no_rename = {
            errno.EXDEV,
            errno.ENOTSUP,
            getattr(errno, "EOPNOTSUPP", errno.ENOTSUP),
            errno.EPERM,
            errno.ENOSYS,
        }
        if exc.errno not in no_rename:
            raise
        print(f"os.replace no disponible ({exc}). Se usa shutil.move.")
        shutil.move(origen, destino)


def download_with_retries(url: str, dest_path: Path, max_retries: int = 3, delay: int = 5) -> None:
    """Descarga url en dest_path. Reintenta y reemplaza el archivo en cada intento."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Descargando {dest_path.name} (intento {attempt}/{max_retries})...")
            with requests.get(url, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                with dest_path.open("wb") as handle:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                    handle.flush()
                    try:
                        os.fsync(handle.fileno())
                    except OSError:
                        pass
            return
        except Exception as exc:
            print(f"Intento {attempt} fallo: {exc}")
            dest_path.unlink(missing_ok=True)
            if attempt == max_retries:
                raise RuntimeError(
                    f"No se pudo descargar {url} despues de {max_retries} intentos."
                ) from exc
            time.sleep(delay)


def verify_zip_integrity(zip_filepath: Path) -> bool:
    """Verifica que el zip sea valido. testzip devuelve None si no hay danos."""
    print(f"Verificando integridad de {zip_filepath.name}...")
    if not zip_filepath.is_file() or not zipfile.is_zipfile(zip_filepath):
        return False
    with zipfile.ZipFile(zip_filepath, "r") as zf:
        return zf.testzip() is None


def tamano_csv_en_zip(zip_path: Path) -> int:
    with zipfile.ZipFile(zip_path) as zf:
        miembros = [
            nombre
            for nombre in zf.namelist()
            if nombre.lower().endswith(".csv")
            and not nombre.endswith("/")
            and "__MACOSX" not in nombre.split("/")
        ]
        if len(miembros) != 1:
            raise ValueError(f"Se esperaba un CSV en {zip_path.name}; hay {len(miembros)}: {miembros}")
        return zf.getinfo(miembros[0]).file_size


def extract_single_csv(zip_path: Path, dest_csv: Path) -> None:
    dest_csv.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        miembros = [
            nombre
            for nombre in zf.namelist()
            if nombre.lower().endswith(".csv")
            and not nombre.endswith("/")
            and "__MACOSX" not in nombre.split("/")
        ]
        if len(miembros) != 1:
            raise ValueError(f"Se esperaba un CSV en {zip_path.name}; hay {len(miembros)}: {miembros}")
        with zf.open(miembros[0]) as origen, dest_csv.open("wb") as destino:
            shutil.copyfileobj(origen, destino, length=8 * 1024 * 1024)
            destino.flush()
            try:
                os.fsync(destino.fileno())
            except OSError:
                pass
    if dest_csv.stat().st_size == 0:
        raise ValueError(f"El CSV extraido quedo vacio: {dest_csv.name}")


def ingest_day(day) -> str:
    """Descarga, extrae y publica sobre /Volumes."""
    nombre = f"AIS_{day.year}_{day.month:02d}_{day.day:02d}"
    zip_final = Path(RAW_ZIPS) / f"{nombre}.zip"
    zip_parcial = Path(STAGING) / f"{nombre}.zip.partial"
    csv_staging = Path(STAGING) / f"{nombre}.csv"
    csv_landing = Path(LANDING) / f"{nombre}.csv"

    for carpeta in (RAW_ZIPS, LANDING, STAGING):
        os.makedirs(carpeta, exist_ok=True)

    if zip_final.is_file() and verify_zip_integrity(zip_final):
        print(f"Zip valido en raw_zips ({zip_final.stat().st_size} bytes). No se descarga.")
        zip_parcial.unlink(missing_ok=True)
    else:
        if zip_final.exists():
            print("El zip en raw_zips no paso la integridad. Se vuelve a bajar.")
            zip_final.unlink()
        url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{day.year}/{nombre}.zip"
        print(f"Fuente: {url}")
        download_with_retries(url, zip_parcial)
        if not verify_zip_integrity(zip_parcial):
            zip_parcial.unlink(missing_ok=True)
            raise ValueError(f"La verificacion de integridad fallo para {nombre}.zip.")
        print(f"Integridad verificada ({zip_parcial.stat().st_size} bytes).")
        mover_en_volume(str(zip_parcial), str(zip_final))

    esperado = tamano_csv_en_zip(zip_final)
    if csv_landing.is_file() and csv_landing.stat().st_size == esperado:
        print(f"El CSV de {day.isoformat()} ya esta en landing ({esperado} bytes).")
        csv_staging.unlink(missing_ok=True)
        return str(csv_landing)

    if not (csv_staging.is_file() and csv_staging.stat().st_size == esperado):
        extract_single_csv(zip_final, csv_staging)
        print(f"CSV extraido ({csv_staging.stat().st_size} bytes).")
        if csv_staging.stat().st_size != esperado:
            raise ValueError(
                f"El CSV mide {csv_staging.stat().st_size} bytes; el zip declara {esperado}."
            )
    else:
        print(f"CSV en staging ya completo ({esperado} bytes).")

    if csv_landing.exists():
        csv_landing.unlink()
    mover_en_volume(str(csv_staging), str(csv_landing))
    print(f"CSV publicado en {csv_landing}")
    return str(csv_landing)


In [ ]:
ingest_day(fecha_dia)
